## 4.5 Gradient Boosting Machine 정리

### GBM 실습 흐름

이번 부분에서는 여러 개의 결정 트리를 차례대로 연결해서 성능을 높이는 GBM을 확인한다.

핵심은 앞 단계 모델이 잘 맞히지 못한 부분을 다음 모델이 이어서 보완한다는 점이다.  
그래서 랜덤 포레스트처럼 한 번에 여러 트리를 만드는 방식이 아니라, 순서대로 모델을 쌓아 올리는 구조에 가깝다.

정리하면 다음과 같다.

| 구분 | 내용 |
|---|---|
| 모델 성격 | 부스팅 기반 앙상블 |
| 기본 학습기 | 주로 결정 트리 사용 |
| 학습 방식 | 이전 예측의 오차를 다음 단계에서 보완 |
| 장점 | 일반적으로 예측 성능이 좋음 |
| 주의점 | 학습 시간이 길고 과적합 관리가 필요함 |

`learning_rate`, `n_estimators`, `max_depth` 같은 값에 따라 성능과 학습 시간이 크게 달라진다.

In [ ]:
# 중복된 피처명이 있으면 뒤에 번호를 붙여 컬럼명을 정리한다.
def get_new_feature_name_df(old_feature_name_df):
    feature_name = old_feature_name_df['column_name'].tolist()
    
    seen = {}
    new_names = []
    for name in feature_name:
        if name in seen:
            seen[name] += 1
            new_names.append(f"{name}_{seen[name]}")
        else:
            seen[name] = 0
            new_names.append(name)
    
    new_df = old_feature_name_df.copy()
    new_df['column_name'] = new_names
    return new_df

In [ ]:
# Human Activity Recognition 데이터 파일을 읽어 학습/테스트 데이터로 나눈다.
import pandas as pd

def get_human_dataset():
    feature_name_df = pd.read_csv('./human_activity/features.txt', sep=r'\s+',
                        header=None, names=['column_index','column_name'])
    
    new_feature_name_df = get_new_feature_name_df(feature_name_df)
    feature_name = new_feature_name_df.iloc[:, 1].values.tolist()
    
    X_train = pd.read_csv('./human_activity/train/X_train.txt', sep=r'\s+', names=feature_name)
    X_test  = pd.read_csv('./human_activity/test/X_test.txt',  sep=r'\s+', names=feature_name)
    
    y_train = pd.read_csv('./human_activity/train/y_train.txt', sep=r'\s+', header=None, names=['action'])
    y_test  = pd.read_csv('./human_activity/test/y_test.txt',  sep=r'\s+', header=None, names=['action'])
    
    return X_train, X_test, y_train, y_test

X_train, X_test, y_train, y_test = get_human_dataset()

In [ ]:
# 기본 GBM 모델을 학습하고 테스트 정확도와 실행 시간을 확인한다.
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score
import time
import warnings
warnings.filterwarnings('ignore')

X_train, X_test, y_train, y_test = get_human_dataset()

start_time = time.time()

gb_clf = GradientBoostingClassifier(random_state=0)
gb_clf.fit(X_train , y_train)
gb_pred = gb_clf.predict(X_test)
gb_accuracy = accuracy_score(y_test, gb_pred)

print('GBM 정확도: {0:.4f}'.format(gb_accuracy))
print("GBM 수행 시간: {0:.1f} 초 ".format(time.time() - start_time))

### GBM에서 자주 조정하는 값

GBM은 기본 설정만으로도 사용할 수 있지만, 실제 성능을 조정하려면 몇 가지 파라미터를 함께 봐야 한다.

| 파라미터 | 의미 | 확인할 점 |
|---|---|---|
| `loss` | 모델이 줄이려고 하는 손실 함수 | 분류 문제에서는 기본 설정을 주로 사용 |
| `learning_rate` | 한 번 업데이트할 때 반영하는 정도 | 작게 잡으면 안정적이지만 느려짐 |
| `n_estimators` | 순서대로 생성할 트리 개수 | 많을수록 오래 걸리고 과적합 가능 |
| `subsample` | 각 단계에서 사용할 데이터 비율 | 1보다 작게 두면 일부 샘플만 사용 |

결국 GBM은 `learning_rate`와 `n_estimators`의 균형이 중요하다.  
학습률을 낮추면 더 많은 트리가 필요하고, 트리 수를 너무 늘리면 시간이 많이 걸린다.

## 4.6 XGBoost 정리

### XGBoost 개념

XGBoost는 GBM 계열 모델을 더 빠르고 안정적으로 사용할 수 있게 만든 알고리즘이다.  
기본 아이디어는 GBM과 비슷하지만, 실제 데이터 분석에서 자주 발생하는 속도 문제와 과적합 문제를 줄이는 기능이 추가되어 있다.

GBM과 비교했을 때 눈에 띄는 특징은 다음과 같다.

| 특징 | 설명 |
|---|---|
| 빠른 학습 | 병렬 처리와 최적화된 연산 구조를 사용 |
| 규제 항 포함 | L1, L2 규제를 통해 과적합을 줄임 |
| 가지치기 | 불필요한 트리 분기를 줄여 모델을 단순화 |
| 결측값 처리 | 내부적으로 결측 방향을 학습할 수 있음 |
| 조기 중단 | 검증 성능이 더 좋아지지 않으면 학습을 멈출 수 있음 |

이후 실습에서는 파이썬 기본 XGBoost 방식과 사이킷런 래퍼 방식을 나누어 사용한다.

In [ ]:
# 설치된 XGBoost 버전을 확인한다.
import xgboost

print(xgboost.__version__)

### XGBoost 기본 API로 유방암 데이터 분류하기

In [ ]:
# 유방암 예제 데이터를 불러와 DataFrame 형태로 정리한다.
import xgboost as xgb
from xgboost import plot_importance
import pandas as pd
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

dataset = load_breast_cancer()
X_features= dataset.data
y_label = dataset.target

cancer_df = pd.DataFrame(data=X_features, columns=dataset.feature_names)
cancer_df['target']= y_label
cancer_df.head(3)

In [ ]:
# 타깃 이름과 클래스별 데이터 개수를 확인한다.
print(dataset.target_names)
print(cancer_df['target'].value_counts())

In [ ]:
# 입력 변수와 정답 값을 나눈 뒤 학습/검증/테스트 데이터로 분리한다.
X_features = cancer_df.iloc[:, :-1]
y_label = cancer_df.iloc[:, -1]

X_train, X_test, y_train, y_test=train_test_split(X_features, y_label,
                                         test_size=0.2, random_state=156 )

X_tr, X_val, y_tr, y_val= train_test_split(X_train, y_train, test_size=0.1, random_state=156 )
print(X_train.shape , X_test.shape)
print(X_tr.shape, X_val.shape)

In [ ]:
# XGBoost 기본 API에서 사용하는 DMatrix 형식으로 데이터를 변환한다.
dtr = xgb.DMatrix(data=X_tr, label=y_tr)
dval = xgb.DMatrix(data=X_val, label=y_val)
dtest = xgb.DMatrix(data=X_test , label=y_test)

In [ ]:
# XGBoost 학습에 사용할 주요 파라미터와 반복 횟수를 지정한다.
params = { 'max_depth':3,
          'eta': 0.05,
          'objective':'binary:logistic',
          'eval_metric':'logloss'
         }
num_rounds = 400

In [ ]:
# 검증 데이터를 함께 넣어 학습 중 성능 변화를 확인한다.
eval_list = [(dtr,'train'),(dval,'eval')]

xgb_model = xgb.train(params = params , dtrain=dtr , num_boost_round=num_rounds , \
                      early_stopping_rounds=50, evals=eval_list )

In [ ]:
# 예측 확률을 기준으로 0과 1 클래스를 결정한다.
pred_probs = xgb_model.predict(dtest)
print('predict( ) 수행 결과값을 10개만 표시, 예측 확률 값으로 표시됨')
print(np.round(pred_probs[:10],3))

preds = [ 1 if x > 0.5 else 0 for x in pred_probs ]
print('예측값 10개만 표시:',preds[:10])

In [ ]:
# 분류 모델의 주요 평가 지표를 한 번에 출력하는 함수를 만든다.
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.metrics import precision_score, recall_score
from sklearn.metrics import f1_score, roc_auc_score

def get_clf_eval(y_test, pred=None, pred_proba=None):
    confusion = confusion_matrix( y_test, pred)
    accuracy = accuracy_score(y_test , pred)
    precision = precision_score(y_test , pred)
    recall = recall_score(y_test , pred)
    f1 = f1_score(y_test,pred)
    roc_auc = roc_auc_score(y_test, pred_proba)
    print('오차 행렬')
    print(confusion)
    print('정확도: {0:.4f}, 정밀도: {1:.4f}, 재현율: {2:.4f},\
    F1: {3:.4f}, AUC:{4:.4f}'.format(accuracy, precision, recall, f1, roc_auc))

In [ ]:
get_clf_eval(y_test , preds, pred_probs)

In [ ]:
# 학습된 XGBoost 모델에서 중요하게 사용된 피처를 시각화한다.
import matplotlib.pyplot as plt
%matplotlib inline

fig, ax = plt.subplots(figsize=(10, 12))
plot_importance(xgb_model, ax=ax)
plt.savefig('p239_xgb_feature_importance.tif', format='tif', dpi=300, bbox_inches='tight')

### 사이킷런 방식의 XGBoost 사용

In [ ]:
# 사이킷런 인터페이스로 XGBoost 분류 모델을 학습한다.
from xgboost import XGBClassifier

xgb_wrapper = XGBClassifier(n_estimators=400, learning_rate=0.1, max_depth=3, eval_metric='logloss')
xgb_wrapper.fit(X_train, y_train, verbose=True)
w_preds = xgb_wrapper.predict(X_test)
w_pred_proba = xgb_wrapper.predict_proba(X_test)[:, 1]

In [ ]:
get_clf_eval(y_test , w_preds, w_pred_proba)

In [ ]:
# 조기 중단 옵션을 포함해 XGBoost 모델을 다시 설정한다.
from xgboost import XGBClassifier

evals = [(X_tr, y_tr), (X_val, y_val)]

xgb_wrapper = XGBClassifier(n_estimators=400, learning_rate=0.1, max_depth=3,
                             early_stopping_rounds=100,
                             eval_metric="logloss")

xgb_wrapper.fit(X_train, y_train,
                eval_set=evals, verbose=True)

ws100_preds = xgb_wrapper.predict(X_test)
ws100_pred_proba = xgb_wrapper.predict_proba(X_test)[:, 1]

In [ ]:
get_clf_eval(y_test , ws100_preds, ws100_pred_proba)

In [ ]:
# 같은 검증 세트를 사용해 조기 중단 조건을 바꾼 뒤 결과를 비교한다.
xgb_wrapper.fit(X_train, y_train, eval_set=evals,verbose=True)

ws10_preds = xgb_wrapper.predict(X_test)
ws10_pred_proba = xgb_wrapper.predict_proba(X_test)[:, 1]
get_clf_eval(y_test , ws10_preds, ws10_pred_proba)

In [ ]:
# 사이킷런 래퍼 모델의 피처 중요도를 그래프로 확인한다.
from xgboost import plot_importance
import matplotlib.pyplot as plt
%matplotlib inline

fig, ax = plt.subplots(figsize=(10, 12))
plot_importance(xgb_wrapper, ax=ax)

## 4.7 LightGBM 정리

### LightGBM 개념

LightGBM은 부스팅 계열 모델 중에서도 학습 속도와 메모리 효율을 강하게 개선한 모델이다.

가장 큰 특징은 트리를 나누는 방식이다.  
일반적인 레벨 중심 분할이 아니라, 손실을 가장 많이 줄일 수 있는 리프를 우선적으로 확장한다.

| 항목 | 내용 |
|---|---|
| 개발 | Microsoft |
| 분할 방식 | Leaf-wise 방식 |
| 장점 | 빠른 학습, 낮은 메모리 사용량 |
| 단점 | 데이터가 적으면 과적합 위험 증가 |
| 활용 | 대용량 정형 데이터에서 자주 사용 |

데이터 수가 충분히 많을 때 특히 유리하지만, 작은 데이터에서는 파라미터 조정이 더 중요하다.

### XGBoost와 LightGBM 파라미터 비교

두 모델은 모두 그래디언트 부스팅 계열이지만, 내부 구현과 트리 성장 방식이 다르다.  
따라서 비슷한 이름의 파라미터도 있고, LightGBM에만 있는 설정도 있다.

| LightGBM | XGBoost | 의미 |
|---|---|---|
| `n_estimators` | `n_estimators` | 생성할 트리 수 |
| `learning_rate` | `learning_rate` | 학습률 |
| `max_depth` | `max_depth` | 트리 깊이 제한 |
| `num_leaves` | - | 리프 노드 최대 개수 |
| `min_child_samples` | `min_child_weight` | 분할 후 필요한 최소 조건 |
| `subsample` | `subsample` | 행 샘플링 비율 |
| `colsample_bytree` | `colsample_bytree` | 열 샘플링 비율 |
| `reg_alpha` | `reg_alpha` | L1 규제 |
| `reg_lambda` | `reg_lambda` | L2 규제 |

LightGBM은 빠른 대신 리프가 깊어질 수 있어서, `num_leaves`, `max_depth`, `min_child_samples`를 같이 확인해야 한다.

In [ ]:
# 설치된 LightGBM 버전을 확인한다.
import lightgbm

print(lightgbm.__version__)

### LightGBM으로 유방암 데이터 학습하기

In [ ]:
# LightGBM 모델 학습을 위해 데이터를 다시 준비하고 모델을 실행한다.
from lightgbm import LGBMClassifier

import pandas as pd
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

dataset = load_breast_cancer()

cancer_df = pd.DataFrame(data=dataset.data, columns=dataset.feature_names)
cancer_df['target']= dataset.target
X_features = cancer_df.iloc[:, :-1]
y_label = cancer_df.iloc[:, -1]

X_train, X_test, y_train, y_test=train_test_split(X_features, y_label, test_size=0.2, random_state=156 )

X_tr, X_val, y_tr, y_val= train_test_split(X_train, y_train, test_size=0.1, random_state=156 )

lgbm_wrapper = LGBMClassifier(n_estimators=400, learning_rate=0.05)

evals = [(X_tr, y_tr), (X_val, y_val)]
lgbm_wrapper.fit(X_tr, y_tr, 
                 eval_set=evals)
preds = lgbm_wrapper.predict(X_test)
pred_proba = lgbm_wrapper.predict_proba(X_test)[:, 1]

In [ ]:
# LightGBM 평가에도 같은 지표 출력 함수를 사용한다.
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.metrics import precision_score, recall_score
from sklearn.metrics import f1_score, roc_auc_score

def get_clf_eval(y_test, pred=None, pred_proba=None):
    confusion = confusion_matrix( y_test, pred)
    accuracy = accuracy_score(y_test , pred)
    precision = precision_score(y_test , pred)
    recall = recall_score(y_test , pred)
    f1 = f1_score(y_test,pred)
    roc_auc = roc_auc_score(y_test, pred_proba)
    print('오차 행렬')
    print(confusion)
    print('정확도: {0:.4f}, 정밀도: {1:.4f}, 재현율: {2:.4f},\
    F1: {3:.4f}, AUC:{4:.4f}'.format(accuracy, precision, recall, f1, roc_auc))

In [ ]:
get_clf_eval(y_test, preds, pred_proba)

In [ ]:
# LightGBM에서 계산한 피처 중요도를 그림 파일로 저장한다.
from lightgbm import plot_importance
import matplotlib.pyplot as plt
%matplotlib inline

fig, ax = plt.subplots(figsize=(10, 12))
plot_importance(lgbm_wrapper, ax=ax)
plt.savefig('lightgbm_feature_importance.tif', format='tif', dpi=300, bbox_inches='tight')

## 4.8 HyperOpt를 이용한 하이퍼파라미터 탐색

### 베이지안 최적화 이해

하이퍼파라미터를 찾는 방법은 여러 가지가 있다.  
단순히 모든 조합을 확인할 수도 있고, 무작위로 일부만 볼 수도 있다.  
베이지안 최적화는 이전 결과를 이용해서 다음에 확인할 후보를 더 똑똑하게 고르는 방식이다.

| 방법 | 방식 | 특징 |
|---|---|---|
| Grid Search | 정해진 조합을 전부 확인 | 정확하지만 느림 |
| Random Search | 임의의 조합을 선택 | 간단하지만 운의 영향이 큼 |
| Bayesian Optimization | 이전 결과를 반영해 다음 후보 선택 | 적은 횟수로 좋은 후보를 찾기 쉬움 |

진행 흐름은 대략 다음과 같다.

1. 몇 개의 조합을 먼저 평가한다.
2. 그 결과를 바탕으로 성능이 좋아질 만한 영역을 추정한다.
3. 다음 후보를 선택해서 다시 평가한다.
4. 이 과정을 반복하면서 더 좋은 조합을 찾는다.

### HyperOpt 사용 메모

HyperOpt는 베이지안 최적화를 파이썬에서 사용할 수 있게 해주는 라이브러리다.  
여기서는 간단한 수식 예제로 동작을 먼저 확인한 뒤, XGBoost 파라미터 탐색에 적용한다.

실행 환경에 따라 패키지 호환 문제가 생길 수 있으므로, 필요한 경우 라이브러리 내부 import 부분을 수정해 사용했다.

In [ ]:
# HyperOpt 패키지를 설치한다.
pip install hyperopt

In [ ]:
# 현재 실행 환경에서 발생한 HyperOpt import 문제를 직접 수정한다.
atpe_path = r'C:\Users\엄지용\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\hyperopt\atpe.py'

with open(atpe_path, 'r', encoding='utf-8') as f:
    content = f.read()

content = content.replace(
    'import pkg_resources',
    'import importlib.metadata as pkg_resources'
)

with open(atpe_path, 'w', encoding='utf-8') as f:
    f.write(content)

print('패치 완료')

In [ ]:
# 간단한 예제 함수에 사용할 x, y 탐색 범위를 지정한다.
from hyperopt import hp

search_space = {'x': hp.quniform('x', -10, 10, 1), 'y': hp.quniform('y', -15, 15, 1) }

In [ ]:
# HyperOpt가 최소화할 목적 함수를 정의한다.
from hyperopt import STATUS_OK

def objective_func(search_space):
    x = search_space['x']
    y = search_space['y']
    retval = x**2 - 20*y
    
    return retval

In [ ]:
# 5번만 탐색해서 대략적인 최적 입력값을 확인한다.
from hyperopt import fmin, tpe, Trials
import numpy as np

trial_val = Trials()

best_01 = fmin(fn=objective_func, space=search_space, algo=tpe.suggest, max_evals=5
               , trials=trial_val, rstate=np.random.default_rng(seed=0))
print('best:', best_01)

In [ ]:
# 탐색 횟수를 늘렸을 때 결과가 어떻게 달라지는지 확인한다.
trial_val = Trials()

best_02 = fmin(fn=objective_func, space=search_space, algo=tpe.suggest, max_evals=20
               , trials=trial_val, rstate=np.random.default_rng(seed=0))
print('best:', best_02)

In [ ]:
# 각 시도에서 계산된 손실값을 확인한다.
print(trial_val.results)

In [ ]:
# 각 시도에서 선택된 입력 변수 값을 확인한다.
print(trial_val.vals)

In [ ]:
# HyperOpt 탐색 결과를 표 형태로 정리한다.
import pandas as pd

losses = [loss_dict['loss'] for loss_dict in trial_val.results]

result_df = pd.DataFrame({'x': trial_val.vals['x'], 'y': trial_val.vals['y'], 'losses': losses})
result_df

### HyperOpt로 XGBoost 파라미터 찾기

In [ ]:
# HyperOpt를 적용할 유방암 데이터를 다시 불러온다.
import pandas as pd
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

dataset = load_breast_cancer()

cancer_df = pd.DataFrame(data=dataset.data, columns=dataset.feature_names)
cancer_df['target']= dataset.target
X_features = cancer_df.iloc[:, :-1]
y_label = cancer_df.iloc[:, -1]

In [ ]:
# 전체 데이터를 학습/테스트로 나누고, 학습 데이터 일부를 검증용으로 분리한다.
X_train, X_test, y_train, y_test=train_test_split(X_features, y_label, test_size=0.2, random_state=156 )

X_tr, X_val, y_tr, y_val= train_test_split(X_train, y_train, test_size=0.1, random_state=156 )

In [ ]:
# XGBoost에서 튜닝할 파라미터의 탐색 범위를 설정한다.
from hyperopt import hp

xgb_search_space = {'max_depth': hp.quniform('max_depth', 5, 20, 1), 
                    'min_child_weight': hp.quniform('min_child_weight', 1, 2, 1),
                    'learning_rate': hp.uniform('learning_rate', 0.01, 0.2),
                    'colsample_bytree': hp.uniform('colsample_bytree', 0.5, 1),
                   }

In [ ]:
# 교차 검증 정확도를 기준으로 HyperOpt 목적 함수를 구성한다.
from sklearn.model_selection import cross_val_score
from xgboost import XGBClassifier
from hyperopt import STATUS_OK

def objective_func(search_space):
    xgb_clf = XGBClassifier(n_estimators=100, max_depth=int(search_space['max_depth']),
                            min_child_weight=int(search_space['min_child_weight']),
                            learning_rate=search_space['learning_rate'],
                            colsample_bytree=search_space['colsample_bytree'],
                            eval_metric='logloss')
    accuracy = cross_val_score(xgb_clf, X_train, y_train, scoring='accuracy', cv=3)
    
    return {'loss':-1 * np.mean(accuracy), 'status': STATUS_OK}

In [ ]:
# 지정한 탐색 공간에서 XGBoost 파라미터 조합을 찾는다.
from hyperopt import fmin, tpe, Trials

trial_val = Trials()
best = fmin(fn=objective_func,
            space=xgb_search_space,
            algo=tpe.suggest,
            max_evals=50,
            trials=trial_val, rstate=np.random.default_rng(seed=9))
print('best:', best)

In [ ]:
# 탐색 결과로 선택된 파라미터 값을 보기 좋게 출력한다.
print('colsample_bytree:{0}, learning_rate:{1}, max_depth:{2}, min_child_weight:{3}'.format(
    round(best['colsample_bytree'], 5), round(best['learning_rate'], 5),
    int(best['max_depth']), int(best['min_child_weight'])))

In [ ]:
# 최종 모델 평가에 사용할 지표 출력 함수를 다시 정의한다.
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.metrics import precision_score, recall_score
from sklearn.metrics import f1_score, roc_auc_score

def get_clf_eval(y_test, pred=None, pred_proba=None):
    confusion = confusion_matrix( y_test, pred)
    accuracy = accuracy_score(y_test , pred)
    precision = precision_score(y_test , pred)
    recall = recall_score(y_test , pred)
    f1 = f1_score(y_test,pred)
    roc_auc = roc_auc_score(y_test, pred_proba)
    print('오차 행렬')
    print(confusion)
    print('정확도: {0:.4f}, 정밀도: {1:.4f}, 재현율: {2:.4f},\
    F1: {3:.4f}, AUC:{4:.4f}'.format(accuracy, precision, recall, f1, roc_auc))

In [ ]:
# 찾은 파라미터를 적용해 최종 XGBoost 모델을 학습하고 평가한다.
xgb_wrapper = XGBClassifier(n_estimators=400,
                            learning_rate=round(best['learning_rate'], 5),
                            max_depth=int(best['max_depth']),
                            min_child_weight=int(best['min_child_weight']),
                            colsample_bytree=round(best['colsample_bytree'], 5)
                           )

evals = [(X_tr, y_tr), (X_val, y_val)]
xgb_wrapper.fit(X_tr, y_tr, 
                eval_set=evals, verbose=True)

preds = xgb_wrapper.predict(X_test)
pred_proba = xgb_wrapper.predict_proba(X_test)[:, 1]

get_clf_eval(y_test, preds, pred_proba)